# Honeybee JSON Import and Visualization with topologic_fast

This notebook demonstrates how to work with Honeybee-style building models.
It covers:

1. Understanding Honeybee JSON (HBJSON) structure
2. Creating room geometries programmatically
3. Adding faces, apertures (windows), and doors
4. Creating shading surfaces
5. Visualizing the complete model

**Note:** Direct HBJSON file parsing (`Honeybee.ByHBJSONPath`) is not yet implemented in topologic_fast.
This notebook demonstrates the concepts using programmatically created geometry that mirrors HBJSON structure.

## Import Required Libraries

In [ ]:
# Import topologic_fast
import topologic_fast as tf

# Import visualization and utility libraries
import plotly.graph_objects as go
import json
import math

print("topologic_fast imported successfully")

## HBJSON Structure Overview

Honeybee JSON files contain:
- **Rooms**: 3D volumes representing thermal zones
- **Faces**: Surfaces (walls, floors, ceilings) with properties
- **Apertures**: Windows and skylights
- **Doors**: Door openings
- **Shades**: External shading surfaces
- **Properties**: Energy and radiance properties

In [ ]:
# Example HBJSON structure (simplified)
example_hbjson = {
    "type": "Model",
    "identifier": "Example_Building",
    "display_name": "Example Building",
    "version": "1.55.2",
    "rooms": [
        {
            "type": "Room",
            "identifier": "Room_1",
            "display_name": "Living Room",
            "faces": [
                {
                    "type": "Face",
                    "identifier": "Face_1",
                    "face_type": "Wall",
                    "boundary": [[0, 0, 0], [5, 0, 0], [5, 0, 3], [0, 0, 3]],
                    "apertures": [
                        {
                            "type": "Aperture",
                            "identifier": "Window_1",
                            "boundary": [[1, 0, 0.8], [4, 0, 0.8], [4, 0, 2.2], [1, 0, 2.2]]
                        }
                    ]
                }
            ]
        }
    ],
    "orphaned_shades": [
        {
            "type": "Shade",
            "identifier": "Overhang_1",
            "geometry": [[0, -0.5, 3], [5, -0.5, 3], [5, 0, 3], [0, 0, 3]]
        }
    ],
    "properties": {
        "energy": {},
        "radiance": {}
    }
}

print("Example HBJSON structure:")
print(json.dumps(example_hbjson, indent=2)[:1000] + "...")

## Create a Honeybee-Style Model

We create a building model that mirrors HBJSON structure.

In [ ]:
class HBRoom:
    """A room with Honeybee-style properties."""
    def __init__(self, identifier, display_name, cell):
        self.identifier = identifier
        self.display_name = display_name
        self.cell = cell  # topologic_fast Cell
        self.faces = []   # List of HBFace
        
class HBFace:
    """A face with Honeybee-style properties."""
    def __init__(self, identifier, face_type, face):
        self.identifier = identifier
        self.face_type = face_type  # Wall, Floor, RoofCeiling, AirBoundary
        self.face = face  # topologic_fast Face
        self.apertures = []  # List of HBAperture
        self.doors = []      # List of HBDoor

class HBAperture:
    """An aperture (window) with Honeybee-style properties."""
    def __init__(self, identifier, face):
        self.identifier = identifier
        self.face = face  # topologic_fast Face

class HBDoor:
    """A door with Honeybee-style properties."""
    def __init__(self, identifier, face, is_glass=False):
        self.identifier = identifier
        self.face = face  # topologic_fast Face
        self.is_glass = is_glass

class HBShade:
    """A shading surface with Honeybee-style properties."""
    def __init__(self, identifier, face):
        self.identifier = identifier
        self.face = face  # topologic_fast Face

class HBModel:
    """A Honeybee-style model container."""
    def __init__(self, identifier):
        self.identifier = identifier
        self.rooms = []
        self.orphaned_faces = []
        self.orphaned_shades = []
        self.orphaned_apertures = []
        self.orphaned_doors = []

print("Honeybee-style classes defined")

## Create a Multi-Room Building

In [ ]:
def create_hb_model():
    """Create a complete Honeybee-style building model."""
    model = HBModel("Example_Building")
    
    # Building dimensions
    room_configs = [
        # (name, x, y, z, width, length, height)
        ("Living_Room", 0, 0, 0, 6, 5, 3),
        ("Kitchen", 6, 0, 0, 4, 5, 3),
        ("Bedroom_1", 0, 5, 0, 5, 4, 3),
        ("Bedroom_2", 5, 5, 0, 5, 4, 3),
        ("Bathroom", 0, 0, 3, 3, 3, 2.5),  # Second floor
    ]
    
    for i, (name, x, y, z, w, l, h) in enumerate(room_configs):
        # Create room cell
        cell = tf.Cell.Box(x, y, z, w, l, h)
        room = HBRoom(f"Room_{i}", name, cell)
        
        # Classify faces
        faces = cell.Faces()
        for j, face in enumerate(faces):
            normal = face.Normal()
            if normal is None:
                continue
            
            nx, ny, nz = normal
            centroid = face.CenterOfMass()
            
            # Classify face type by normal
            if abs(nz) > 0.9:
                if nz > 0:
                    face_type = "RoofCeiling"
                else:
                    face_type = "Floor"
            else:
                face_type = "Wall"
            
            hb_face = HBFace(f"{name}_Face_{j}", face_type, face)
            
            # Add windows to exterior walls (simplified check)
            if face_type == "Wall":
                # Check if this is an exterior wall (on building perimeter)
                cx, cy, cz = centroid
                is_exterior = (
                    abs(cx) < 0.1 or abs(cy) < 0.1 or
                    abs(cx - 10) < 0.1 or abs(cy - 9) < 0.1
                )
                
                if is_exterior and face.Area() > 3:  # Only large walls
                    # Create aperture (window) - 40% of wall
                    aperture = create_aperture(face, 0.4)
                    if aperture:
                        hb_aperture = HBAperture(f"{name}_Window_{j}", aperture)
                        hb_face.apertures.append(hb_aperture)
            
            room.faces.append(hb_face)
        
        model.rooms.append(room)
    
    # Add orphaned shading surfaces (overhangs)
    shade_configs = [
        # (name, x, y, z, width, depth)
        ("Overhang_South", 0, -0.5, 3, 10, 0.5),
        ("Overhang_North", 0, 9, 3, 10, 0.5),
    ]
    
    for name, x, y, z, w, d in shade_configs:
        v1 = tf.Vertex.ByCoordinates(x, y, z)
        v2 = tf.Vertex.ByCoordinates(x + w, y, z)
        v3 = tf.Vertex.ByCoordinates(x + w, y + d, z)
        v4 = tf.Vertex.ByCoordinates(x, y + d, z)
        wire = tf.Wire.ByVertices([v1, v2, v3, v4], close=True)
        shade_face = tf.Face.ByWire(wire)
        model.orphaned_shades.append(HBShade(name, shade_face))
    
    return model

def create_aperture(face, ratio):
    """Create a scaled aperture from a face."""
    centroid = face.CenterOfMass()
    vertices = face.Vertices()
    
    if len(vertices) < 3:
        return None
    
    scale = math.sqrt(ratio)
    cx, cy, cz = centroid
    
    scaled_vertices = []
    for v in vertices:
        vx, vy, vz = v.Coordinates()
        new_x = cx + (vx - cx) * scale
        new_y = cy + (vy - cy) * scale
        new_z = cz + (vz - cz) * scale
        scaled_vertices.append(tf.Vertex.ByCoordinates(new_x, new_y, new_z))
    
    wire = tf.Wire.ByVertices(scaled_vertices, close=True)
    return tf.Face.ByWire(wire)

# Create the model
hb_model = create_hb_model()

print(f"Created Honeybee Model: {hb_model.identifier}")
print(f"\nRooms: {len(hb_model.rooms)}")
for room in hb_model.rooms:
    num_apertures = sum(len(f.apertures) for f in room.faces)
    print(f"  - {room.display_name}: {len(room.faces)} faces, {num_apertures} apertures")

print(f"\nOrphaned Shades: {len(hb_model.orphaned_shades)}")
for shade in hb_model.orphaned_shades:
    print(f"  - {shade.identifier}")

## Analyze the Model

In [ ]:
def analyze_hb_model(model):
    """Compute statistics for a Honeybee model."""
    stats = {
        'rooms': [],
        'total_volume': 0,
        'total_floor_area': 0,
        'total_wall_area': 0,
        'total_aperture_area': 0,
        'total_shade_area': 0,
    }
    
    for room in model.rooms:
        room_stats = {
            'name': room.display_name,
            'volume': room.cell.Volume(),
            'floor_area': 0,
            'wall_area': 0,
            'aperture_area': 0,
        }
        
        for hb_face in room.faces:
            area = hb_face.face.Area()
            
            if hb_face.face_type == "Floor":
                room_stats['floor_area'] += area
            elif hb_face.face_type == "Wall":
                room_stats['wall_area'] += area
            
            for aperture in hb_face.apertures:
                room_stats['aperture_area'] += aperture.face.Area()
        
        stats['rooms'].append(room_stats)
        stats['total_volume'] += room_stats['volume']
        stats['total_floor_area'] += room_stats['floor_area']
        stats['total_wall_area'] += room_stats['wall_area']
        stats['total_aperture_area'] += room_stats['aperture_area']
    
    for shade in model.orphaned_shades:
        stats['total_shade_area'] += shade.face.Area()
    
    return stats

# Analyze the model
stats = analyze_hb_model(hb_model)

print("Model Statistics:")
print("=" * 50)
print(f"Total Volume: {stats['total_volume']:.2f} m^3")
print(f"Total Floor Area: {stats['total_floor_area']:.2f} m^2")
print(f"Total Wall Area: {stats['total_wall_area']:.2f} m^2")
print(f"Total Aperture Area: {stats['total_aperture_area']:.2f} m^2")
print(f"Total Shade Area: {stats['total_shade_area']:.2f} m^2")

if stats['total_wall_area'] > 0:
    wwr = stats['total_aperture_area'] / stats['total_wall_area']
    print(f"\nWindow-to-Wall Ratio: {wwr:.1%}")

print("\nRoom Details:")
print("-" * 50)
for room_stats in stats['rooms']:
    print(f"{room_stats['name']}:")
    print(f"  Volume: {room_stats['volume']:.2f} m^3")
    print(f"  Floor Area: {room_stats['floor_area']:.2f} m^2")
    print(f"  Wall Area: {room_stats['wall_area']:.2f} m^2")
    print(f"  Aperture Area: {room_stats['aperture_area']:.2f} m^2")

## Visualize the Honeybee Model

In [ ]:
def create_face_mesh(face):
    """Extract mesh data from a face."""
    vertices = face.Vertices()
    if len(vertices) < 3:
        return None, None
    
    coords = [v.Coordinates() for v in vertices]
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    z = [c[2] for c in coords]
    
    i_idx, j_idx, k_idx = [], [], []
    for idx in range(1, len(coords) - 1):
        i_idx.append(0)
        j_idx.append(idx)
        k_idx.append(idx + 1)
    
    return (x, y, z), (i_idx, j_idx, k_idx)

def visualize_hb_model(model):
    """Create interactive 3D visualization of Honeybee model."""
    fig = go.Figure()
    
    # Color mapping for face types
    face_colors = {
        "Wall": "lightgray",
        "Floor": "brown",
        "RoofCeiling": "tan"
    }
    
    # Room colors
    room_colors = [
        "rgba(255, 182, 193, 0.3)",  # Light pink
        "rgba(173, 216, 230, 0.3)",  # Light blue
        "rgba(144, 238, 144, 0.3)",  # Light green
        "rgba(255, 218, 185, 0.3)",  # Peach
        "rgba(221, 160, 221, 0.3)",  # Plum
    ]
    
    # Add rooms
    for i, room in enumerate(model.rooms):
        # Combine all face meshes for the room
        all_x, all_y, all_z = [], [], []
        all_i, all_j, all_k = [], [], []
        
        for hb_face in room.faces:
            coords, indices = create_face_mesh(hb_face.face)
            if coords is None:
                continue
            
            offset = len(all_x)
            all_x.extend(coords[0])
            all_y.extend(coords[1])
            all_z.extend(coords[2])
            all_i.extend([idx + offset for idx in indices[0]])
            all_j.extend([idx + offset for idx in indices[1]])
            all_k.extend([idx + offset for idx in indices[2]])
        
        if all_x:
            color = room_colors[i % len(room_colors)]
            fig.add_trace(go.Mesh3d(
                x=all_x, y=all_y, z=all_z,
                i=all_i, j=all_j, k=all_k,
                color=color.replace('0.3', '0.4'),
                opacity=0.4,
                name=room.display_name,
                flatshading=True,
                showlegend=True
            ))
    
    # Add apertures (windows)
    apt_x, apt_y, apt_z = [], [], []
    apt_i, apt_j, apt_k = [], [], []
    
    for room in model.rooms:
        for hb_face in room.faces:
            for aperture in hb_face.apertures:
                coords, indices = create_face_mesh(aperture.face)
                if coords is None:
                    continue
                
                offset = len(apt_x)
                apt_x.extend(coords[0])
                apt_y.extend(coords[1])
                apt_z.extend(coords[2])
                apt_i.extend([idx + offset for idx in indices[0]])
                apt_j.extend([idx + offset for idx in indices[1]])
                apt_k.extend([idx + offset for idx in indices[2]])
    
    if apt_x:
        fig.add_trace(go.Mesh3d(
            x=apt_x, y=apt_y, z=apt_z,
            i=apt_i, j=apt_j, k=apt_k,
            color='lightblue',
            opacity=0.7,
            name='Apertures (Windows)',
            flatshading=True,
            showlegend=True
        ))
    
    # Add shading surfaces
    shade_x, shade_y, shade_z = [], [], []
    shade_i, shade_j, shade_k = [], [], []
    
    for shade in model.orphaned_shades:
        coords, indices = create_face_mesh(shade.face)
        if coords is None:
            continue
        
        offset = len(shade_x)
        shade_x.extend(coords[0])
        shade_y.extend(coords[1])
        shade_z.extend(coords[2])
        shade_i.extend([idx + offset for idx in indices[0]])
        shade_j.extend([idx + offset for idx in indices[1]])
        shade_k.extend([idx + offset for idx in indices[2]])
    
    if shade_x:
        fig.add_trace(go.Mesh3d(
            x=shade_x, y=shade_y, z=shade_z,
            i=shade_i, j=shade_j, k=shade_k,
            color='green',
            opacity=0.8,
            name='Shading Surfaces',
            flatshading=True,
            showlegend=True
        ))
    
    # Update layout
    fig.update_layout(
        title=f'Honeybee Model: {model.identifier}',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.0)
            )
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

# Visualize the model
fig = visualize_hb_model(hb_model)
fig.show()

## Export Model Statistics

In [ ]:
def model_to_dict(model, stats):
    """Convert model to a dictionary structure similar to HBJSON."""
    result = {
        "identifier": model.identifier,
        "type": "Model",
        "statistics": {
            "total_volume_m3": round(stats['total_volume'], 2),
            "total_floor_area_m2": round(stats['total_floor_area'], 2),
            "total_wall_area_m2": round(stats['total_wall_area'], 2),
            "total_aperture_area_m2": round(stats['total_aperture_area'], 2),
            "total_shade_area_m2": round(stats['total_shade_area'], 2),
            "window_to_wall_ratio": round(stats['total_aperture_area'] / stats['total_wall_area'], 3) if stats['total_wall_area'] > 0 else 0
        },
        "rooms": []
    }
    
    for room, room_stats in zip(model.rooms, stats['rooms']):
        room_dict = {
            "identifier": room.identifier,
            "display_name": room.display_name,
            "volume_m3": round(room_stats['volume'], 2),
            "floor_area_m2": round(room_stats['floor_area'], 2),
            "wall_area_m2": round(room_stats['wall_area'], 2),
            "aperture_area_m2": round(room_stats['aperture_area'], 2),
            "num_faces": len(room.faces),
            "num_apertures": sum(len(f.apertures) for f in room.faces)
        }
        result["rooms"].append(room_dict)
    
    result["orphaned_shades"] = [
        {
            "identifier": shade.identifier,
            "area_m2": round(shade.face.Area(), 2)
        }
        for shade in model.orphaned_shades
    ]
    
    return result

# Export to dictionary
model_dict = model_to_dict(hb_model, stats)

print("Model Summary (JSON format):")
print(json.dumps(model_dict, indent=2))

## Visualization by Face Type

In [ ]:
def visualize_by_face_type(model):
    """Create visualization grouped by face type."""
    fig = go.Figure()
    
    # Collect faces by type
    faces_by_type = {
        "Wall": [],
        "Floor": [],
        "RoofCeiling": []
    }
    
    for room in model.rooms:
        for hb_face in room.faces:
            if hb_face.face_type in faces_by_type:
                faces_by_type[hb_face.face_type].append(hb_face.face)
    
    colors = {
        "Wall": "lightgray",
        "Floor": "saddlebrown",
        "RoofCeiling": "tan"
    }
    
    # Add traces for each face type
    for face_type, faces in faces_by_type.items():
        all_x, all_y, all_z = [], [], []
        all_i, all_j, all_k = [], [], []
        
        for face in faces:
            coords, indices = create_face_mesh(face)
            if coords is None:
                continue
            
            offset = len(all_x)
            all_x.extend(coords[0])
            all_y.extend(coords[1])
            all_z.extend(coords[2])
            all_i.extend([idx + offset for idx in indices[0]])
            all_j.extend([idx + offset for idx in indices[1]])
            all_k.extend([idx + offset for idx in indices[2]])
        
        if all_x:
            fig.add_trace(go.Mesh3d(
                x=all_x, y=all_y, z=all_z,
                i=all_i, j=all_j, k=all_k,
                color=colors.get(face_type, 'gray'),
                opacity=0.6,
                name=f"{face_type} ({len(faces)} faces)",
                flatshading=True,
                showlegend=True
            ))
    
    # Update layout
    fig.update_layout(
        title='Building Faces by Type',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900,
        height=700
    )
    
    return fig

# Visualize by face type
fig = visualize_by_face_type(hb_model)
fig.show()

## Summary

This notebook demonstrated Honeybee JSON concepts using topologic_fast:

1. **Model Structure**: Created rooms, faces, apertures, and shading surfaces
2. **Face Classification**: Categorized faces as Walls, Floors, or RoofCeilings
3. **Apertures**: Added window openings to exterior walls
4. **Analysis**: Computed areas, volumes, and ratios
5. **Visualization**: Created interactive 3D views with color-coding
6. **Export**: Generated JSON summaries of model statistics

### Features Not Yet Implemented in topologic_fast

- `Honeybee.ByHBJSONPath()` - Import HBJSON files directly
- `Honeybee.ModelByTopology()` - Create Honeybee model from topology
- `Honeybee.ExportToHBJSON()` - Export to HBJSON format
- Energy and radiance property assignment
- Room adjacency detection
- Face boundary condition assignment

These features are planned for future releases.

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")